### Getting Started with Local AI

Let's get your computer set up to test out product ideas without needing any API keys. :) There are plenty of freely available local Ai models that can help you develop minimal MVPs without paying for a single token.

1. Get [ollama](https://ollama.com/) set up and running. You might also want to [download the desktop-based software](https://ollama.com/download) in case you want to test out prompts outside of these notebooks.

2. Get at least one [llamafile](https://github.com/Mozilla-Ocho/llamafile) running.

3. Set up a workflow for loading and saving prompts and responses.

In [ ]:
import ollama

In [ ]:
for model in ollama.list().models:
    print(model.model)

### Explore and Download a Model

Check out and download at least one model from [the list of available models](https://ollama.com/library?sort=popular). Note: if you do not have a GPU or M1+ chip, you probably should start with the smaller models (i.e. 4b or less).

In [ ]:
model_name = 'gemma3:1b'

In [ ]:
ollama.pull(model_name)

### Prompt a Model

Try it out!

In [ ]:
test_prompt = "ummmm... anyone there?"

In [ ]:
messages = [{
    'role': 'user',
    'content': test_prompt
}]

In [ ]:
response = ollama.chat(
    model=model_name,
    messages=messages
)

In [ ]:
response

In [ ]:
response.__dict__

In [ ]:
print(response.message.content)

In [ ]:
del ollama

### Let's get llamafile going

1. Choose a file from the [downloads page](https://github.com/Mozilla-Ocho/llamafile?tab=readme-ov-file#other-example-llamafiles).
2. Move it to this folder or a path that you can easily access.
3. Open a terminal.
4. Make the file executable: chmod +x MODEL_FILE_PATH
5. Run the file, try the version two by adding --server --v2 to the end of the file name

You should see a browser open. It will have lots of options to enter a prompt. You can test it in the browser at the bottom of the page that opened.

In [ ]:
from openai import OpenAI

In [ ]:
client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=messages
)

In [ ]:
completion

In [ ]:
completion.__dict__

In [ ]:
completion.choices[0].message.content

Well, we can see not all models are built alike ;)

### Building some ways to load templates and save chats

We will set up:

1. a very simple meta prompt template
2. an example prompt
3. an example conversation

The point is to give us some conventions around saving our work as we progress and building best practices to save conversations for later evaluation and review.

In [ ]:
import json

In [ ]:
meta_prompt_template = """
{role_description}

{task_description}

{rules}

{examples}

{context}
"""


In [ ]:
role = "You are a helpful assistant who makes tasty recipes."
task = """You use regular pantry items and things everyone has 
in their kitchen and help them make new, inventive recipes with those ingredients."""
rules = "You do not use fancy ingredients that people might not have access to use."
examples = "An example recipe would be a French omlette with spruced up canned beans and a side salad with a quirky vinegrette."
context = "You take inspiration from Nigella Lawson."

In [ ]:
meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [ ]:
with open('data/templates/meta_prompt_template', 'w') as mpt:
    mpt.write(meta_prompt_template)

In [ ]:
with open('data/meta_prompts/pantry_chef_prompt', 'w') as cpt:
    cpt.write(meta_prompt_template.format(
        role_description=role,
        task_description=task,
        rules=rules,
        examples=examples,
        context=context))

In [ ]:
meta_prompt = meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [ ]:
conversation = [
    {'role': 'system', 
     'content': meta_prompt},
    {'role': 'user',
     'content': """Help! I only have canned tomatoes, some chicken and an onion.... 
     what can I make for dinner?"""},
]

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=conversation
)

In [ ]:
completion

In [ ]:
completion.choices[0].message.content

In [ ]:
conversation.append({
    'role': 'agent',
    'content': completion.choices[0].message.content
})

In [ ]:
with open('data/traces/chef_chicken_pasta.json', 'w') as chef_chat:
    json.dump(conversation, chef_chat)

In [ ]:
from pprint import pprint

In [ ]:
pprint(conversation)